<a href="https://colab.research.google.com/github/tryingtolearn1113/ai-projects/blob/main/LawVectorEmbedding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# Cell 1
# Environment
# ==========================================

!pip -q install -U \
sentence-transformers \
chromadb \
kiwipiepy \
rank-bm25 \
requests \
tqdm \
orjson

import os
import re
import gc
import json
import shutil
import pickle
import requests

from pathlib import Path
from dataclasses import dataclass, field, asdict

import numpy as np
import torch

from tqdm.auto import tqdm

from sentence_transformers import SentenceTransformer
from kiwipiepy import Kiwi
from chromadb import PersistentClient

# ---------------------------------------
# Project Directory
# ---------------------------------------

ROOT = Path("/content/legal_db_v4")

LAW_DIR = ROOT / "laws"

CACHE_DIR = ROOT / "cache"

DB_DIR = ROOT / "chromadb"

INDEX_DIR = ROOT / "index"

for d in [
    ROOT,
    LAW_DIR,
    CACHE_DIR,
    DB_DIR,
    INDEX_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

print("ROOT :", ROOT)

# ---------------------------------------
# Device
# ---------------------------------------

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("DEVICE :", DEVICE)

# ---------------------------------------
# Embedding Model
# ---------------------------------------

print("Loading BGE-M3...")

embedding_model = SentenceTransformer(
    "BAAI/bge-m3",
    device=DEVICE,
)

embedding_model.max_seq_length = 1024

print("Embedding model loaded.")

# ---------------------------------------
# Kiwi
# ---------------------------------------

print("Loading Kiwi...")

kiwi = Kiwi()

print("Kiwi loaded.")

# ---------------------------------------
# Chroma
# ---------------------------------------

client = PersistentClient(
    path=str(DB_DIR)
)

COLLECTION_NAME = "legal_v4"

existing = [
    c.name
    for c in client.list_collections()
]

if COLLECTION_NAME in existing:
    client.delete_collection(COLLECTION_NAME)

collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={
        "hnsw:space": "cosine"
    }
)

print("Chroma ready.")

# ---------------------------------------
# Random Seed
# ---------------------------------------

torch.manual_seed(42)
np.random.seed(42)

# ---------------------------------------
# Dataclass
# ---------------------------------------

@dataclass
class Article:

    id: str

    law_name: str

    part: str = ""

    chapter: str = ""

    section: str = ""

    subsection: str = ""

    article: str = ""

    title: str = ""

    clauses: list = field(default_factory=list)

    display_text: str = ""

    search_text: str = ""

    references: list = field(default_factory=list)

    keywords: list = field(default_factory=list)

    kiwi_tokens: list = field(default_factory=list)

    metadata: dict = field(default_factory=dict)

print()

print("=" * 60)
print("Environment Ready")
print("=" * 60)

ROOT : /content/legal_db_v4
DEVICE : cuda
Loading BGE-M3...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Embedding model loaded.
Loading Kiwi...
Kiwi loaded.
Chroma ready.

Environment Ready


In [ ]:
# ==========================================
# Cell 2
# Download legalize-kr Laws
# ==========================================

import time

# -------------------------------------------------------
# 변호사시험 대상 법령
# -------------------------------------------------------

TARGET_LAWS = [

    "대한민국헌법",

    "민법",
    "형법",
    "상법",

    "민사소송법",
    "형사소송법",

    "행정소송법",
    "행정절차법",
    "행정기본법",

    "지방자치법",

    "국회법",

    "공익사업을 위한 토지 등의 취득 및 보상에 관한 법률",

    "부동산등기법",

    "민사집행법",

    "어음법",

    "상표법",

    "특허법",

    "저작권법",

    "부정경쟁방지 및 영업비밀보호에 관한 법률",

    "환경영향평가법",

    "국제물품매매계약에 관한 유엔협약",

    "주택임대차보호법",

    "상가건물임대차보호법",

    "국가배상법",

    "부동산실권리자명의등기에관한법률",

    "가등기담보등에관한법률",

    "집합건물의소유및관리에관한법률",

    "특정범죄가중처벌등에관한법률",

    "특정경제범죄가중처벌등에관한법률",

    "폭력행위등처벌에관한법률",

    "성폭력범죄의처벌등에관한특례법",

    "교통사고처리특례법",

    "도로교통법",

    "근로기준법",

    "노동조합 및 노동관계조정법",

    "국세기본법",

    "소득세법",

    "부가가치세법",

    "독점규제 및 공정거래에 관한 법률",

    "할부거래에 관한 법률",

    "약관의 규제에 관한 법률",

    "폐기물관리법",

    "물환경보전법",

    "국제사법"

]

print(f"Target Laws : {len(TARGET_LAWS)}")

# -------------------------------------------------------
# github
# -------------------------------------------------------

BASE = "https://raw.githubusercontent.com/legalize-kr/legalize-kr/main/kr"

CANDIDATES = [

    "헌법.md",
    "법률.md",
    "조약.md",

]

session = requests.Session()

session.headers.update({

    "User-Agent":"Mozilla/5.0"

})

# -------------------------------------------------------
# download
# -------------------------------------------------------

def download_law(law_name):

    folder = law_name.replace(" ","")

    for filename in CANDIDATES:

        url = f"{BASE}/{folder}/{filename}"

        for retry in range(3):

            try:

                r = session.get(
                    url,
                    timeout=20
                )

                if r.status_code != 200:
                    continue

                text = r.text.strip()

                if len(text) < 100:
                    continue

                save_path = LAW_DIR / f"{law_name}.md"

                with open(
                    save_path,
                    "w",
                    encoding="utf8"
                ) as f:

                    f.write(text)

                return True

            except Exception:

                time.sleep(1)

    return False

# -------------------------------------------------------
# run
# -------------------------------------------------------

failed = []

print()

print("="*60)

print("Downloading...")

print("="*60)

for law in tqdm(TARGET_LAWS):

    ok = download_law(law)

    if not ok:

        failed.append(law)

print()

print("="*60)

print("Finished")

print("="*60)

files = sorted(LAW_DIR.glob("*.md"))

print()

print("Downloaded :", len(files))

print("Failed :", len(failed))

if failed:

    print()

    print("Failed Laws")

    for x in failed:

        print("-",x)

# -------------------------------------------------------
# preview
# -------------------------------------------------------

print()

print("="*60)

print("Preview")

print("="*60)

sample = files[0]

print(sample.name)

print()

with open(sample,encoding="utf8") as f:

    print(f.read()[:1200])


Target Laws : 44

Downloading...


  0%|          | 0/44 [00:00<?, ?it/s]


Finished

Downloaded : 43
Failed : 1

Failed Laws
- 국제물품매매계약에 관한 유엔협약

Preview
가등기담보등에관한법률.md

---
제목: 가등기담보 등에 관한 법률
법령MST: 188538
법령ID: '001257'
법령구분: 법률
법령구분코드: ''
소관부처:
- 법무부
공포일자: 2016-12-27
공포번호: '14474'
시행일자: 2017-03-28
법령분야: ''
상태: 시행
출처: https://www.law.go.kr/법령/가등기담보등에관한법률
첨부파일: []
---

# 가등기담보 등에 관한 법률

##### 제1조 (목적)

이 법은 차용물(借用物)의 반환에 관하여 차주(借主)가 차용물을 갈음하여 다른 재산권을 이전할 것을 예약할 때 그 재산의 예약 당시 가액(價額)이 차용액(借用額)과 이에 붙인 이자를 합산한 액수를 초과하는 경우에 이에 따른 담보계약(擔保契約)과 그 담보의 목적으로 마친 가등기(假登記) 또는 소유권이전등기(所有權移轉登記)의 효력을 정함을 목적으로 한다.

##### 제2조 (정의)

이 법에서 사용하는 용어의 뜻은 다음과 같다.

  1\. "담보계약"이란 「민법」 제608조에 따라 그 효력이 상실되는 대물반환(代物返還)의 예약[환매(還買), 양도담보(讓渡擔保) 등 명목(名目)이 어떠하든 그 모두를 포함한다]에 포함되거나 병존(竝存)하는 채권담보(債權擔保) 계약을 말한다.
  2\. "채무자등"이란 다음 각 목의 자를 말한다.
    가\. 채무자
    나\. 담보가등기목적 부동산의 물상보증인(物上保證人)
    다\. 담보가등기 후 소유권을 취득한 제삼자
  3\. "담보가등기(擔保假登記)"란 채권담보의 목적으로 마친 가등기를 말한다.
  4\. "강제경매등"이란 강제경매(强制競賣)와 담보권의 실행 등을 위한 경매를 말한다.
  5\. "후순위권리자(後順位權利者)"란 담보가등기 후에 등기된 저당권자ㆍ전세권자 및 담보가등기권리자를 말한다.

##### 제3조 (담보권 실행의 

In [ ]:
# ==========================================
# Cell 3-A
# Markdown Parser Core
# ==========================================

import re
from dataclasses import dataclass, field
from pathlib import Path


# --------------------------------------------------
# Regex
# --------------------------------------------------

PART_RE = re.compile(
    r"^#+\s*(제\s*\d+\s*편.*)"
)

CHAPTER_RE = re.compile(
    r"^#+\s*(제\s*\d+\s*장.*)"
)

SECTION_RE = re.compile(
    r"^#+\s*(제\s*\d+\s*절.*)"
)

SUBSECTION_RE = re.compile(
    r"^#+\s*(제\s*\d+\s*관.*)"
)

ARTICLE_RE = re.compile(
    r"^#+\s*(제\s*\d+\s*조(?:의\s*\d+)?)"
    r"(?:\s*\((.*?)\))?"
)


CLAUSE_RE = re.compile(
    r"^([①②③④⑤⑥⑦⑧⑨⑩])\s*(.*)"
)


REFERENCE_RE = re.compile(
    r"제\s*\d+\s*조(?:의\s*\d+)?"
)



# --------------------------------------------------
# Data Object
# --------------------------------------------------

@dataclass
class ArticleObject:

    id: str

    law_name: str

    part: str = ""

    chapter: str = ""

    section: str = ""

    subsection: str = ""

    article: str = ""

    title: str = ""

    clauses: list = field(default_factory=list)

    body: str = ""

    references: list = field(default_factory=list)



# --------------------------------------------------
# Utility
# --------------------------------------------------

def normalize_article(text):

    return (
        text
        .replace(" ","")
        .strip()
    )



def make_article_id(
    law_name,
    article
):

    number = re.sub(
        r"[^\d의]",
        "",
        article
    )

    return (
        f"{law_name}_{number}"
    )



def extract_reference(text):

    result = set()

    for x in REFERENCE_RE.findall(text):

        result.add(
            x.replace(" ","")
        )

    return sorted(result)



print("Parser Core Loaded")

# ==========================================
# Cell 3-B
# Single Markdown Parser
# ==========================================


def split_clauses(lines):
    """
    조문 본문을 항 단위로 분리

    입력:
        [
          "① 사람을 기망하여...",
          "② 전항의..."
        ]

    출력:
        [
          {
            "no":1,
            "text":"..."
          },
          ...
        ]
    """

    clauses = []

    current = None


    for line in lines:

        line = line.strip()

        if not line:
            continue


        m = CLAUSE_RE.match(line)


        if m:

            if current:

                clauses.append(
                    current
                )


            symbol = m.group(1)

            number = (
                "①②③④⑤⑥⑦⑧⑨⑩"
                .index(symbol)
                + 1
            )


            current = {

                "no": number,

                "symbol": symbol,

                "text": m.group(2).strip()

            }


        else:

            if current:

                current["text"] += " " + line



    if current:

        clauses.append(
            current
        )


    return clauses



# --------------------------------------------------
# Parse one markdown
# --------------------------------------------------

def parse_single_markdown(
    filepath:Path
):


    law_name = filepath.stem


    with open(
        filepath,
        encoding="utf8"
    ) as f:

        lines = [
            x.rstrip()
            for x in f.readlines()
        ]



    articles = []


    current_part = ""

    current_chapter = ""

    current_section = ""

    current_subsection = ""


    current_article = None

    current_title = ""

    buffer = []



    def save_article():

        nonlocal current_article
        nonlocal current_title
        nonlocal buffer


        if current_article is None:

            return



        body = "\n".join(
            buffer
        ).strip()



        clauses = split_clauses(
            buffer
        )



        refs = extract_reference(
            body
        )



        obj = ArticleObject(

            id=make_article_id(
                law_name,
                current_article
            ),

            law_name=law_name,

            part=current_part,

            chapter=current_chapter,

            section=current_section,

            subsection=current_subsection,

            article=current_article,

            title=current_title,

            clauses=clauses,

            body=body,

            references=refs

        )


        articles.append(
            obj
        )


        buffer = []



    # --------------------------------------------------
    # line loop
    # --------------------------------------------------

    for line in lines:


        # 빈 줄

        if not line.strip():

            continue



        # 부칙 이후 제거

        if (
            "부칙" in line
            and line.startswith("#")
        ):

            break



        # 편

        m = PART_RE.match(line)

        if m:

            current_part = m.group(1)

            continue



        # 장

        m = CHAPTER_RE.match(line)

        if m:

            current_chapter = m.group(1)

            continue



        # 절

        m = SECTION_RE.match(line)

        if m:

            current_section = m.group(1)

            continue



        # 관

        m = SUBSECTION_RE.match(line)

        if m:

            current_subsection = m.group(1)

            continue



        # 조문

        m = ARTICLE_RE.match(line)


        if m:


            save_article()


            current_article = normalize_article(
                m.group(1)
            )


            current_title = (
                m.group(2).strip()
                if m.group(2)
                else ""
            )


            buffer = []


            continue



        # 본문

        if current_article:

            buffer.append(
                line.strip()
            )



    # 마지막 저장

    save_article()


    return articles



print("Single Markdown Parser Ready")

# ==========================================
# Cell 3-C
# Parse All Markdown Laws
# ==========================================


from tqdm.auto import tqdm


# --------------------------------------------------
# Parse all files
# --------------------------------------------------

ALL_ARTICLES = []

FAILED_FILES = []


law_files = sorted(
    LAW_DIR.glob("*.md")
)


print()
print("="*60)
print("Parsing Laws")
print("="*60)

print(
    "Files:",
    len(law_files)
)



for filepath in tqdm(law_files):

    try:

        articles = parse_single_markdown(
            filepath
        )

        ALL_ARTICLES.extend(
            articles
        )


    except Exception as e:

        FAILED_FILES.append(
            {
                "file":filepath.name,
                "error":str(e)
            }
        )



print()

print("="*60)

print("Parsing Finished")

print("="*60)

print()

print(
    "Total Articles:",
    len(ALL_ARTICLES)
)


print(
    "Failed Files:",
    len(FAILED_FILES)
)



if FAILED_FILES:

    print()

    print("FAILED")

    for x in FAILED_FILES:

        print(
            x["file"],
            x["error"]
        )



# --------------------------------------------------
# Save cache
# --------------------------------------------------

import pickle


with open(
    CACHE_DIR / "articles.pkl",
    "wb"
) as f:

    pickle.dump(
        ALL_ARTICLES,
        f
    )



print()

print(
    "Saved:",
    CACHE_DIR / "articles.pkl"
)



# --------------------------------------------------
# Sample
# --------------------------------------------------

print()

print("="*60)

print("Sample Article")

print("="*60)


sample = ALL_ARTICLES[0]


print(
    sample
)



print()

print("ID")
print(sample.id)


print()

print("LAW")
print(sample.law_name)


print()

print("ARTICLE")
print(sample.article)


print()

print("TITLE")
print(sample.title)


print()

print("CLAUSES")

for c in sample.clauses[:3]:

    print(
        c
    )


print()

print("REFERENCES")

print(
    sample.references
)

Parser Core Loaded
Single Markdown Parser Ready

Parsing Laws
Files: 43


  0%|          | 0/43 [00:00<?, ?it/s]


Parsing Finished

Total Articles: 7852
Failed Files: 0

Saved: /content/legal_db_v4/cache/articles.pkl

Sample Article
ArticleObject(id='가등기담보등에관한법률_1', law_name='가등기담보등에관한법률', part='', chapter='', section='', subsection='', article='제1조', title='목적', clauses=[], body='이 법은 차용물(借用物)의 반환에 관하여 차주(借主)가 차용물을 갈음하여 다른 재산권을 이전할 것을 예약할 때 그 재산의 예약 당시 가액(價額)이 차용액(借用額)과 이에 붙인 이자를 합산한 액수를 초과하는 경우에 이에 따른 담보계약(擔保契約)과 그 담보의 목적으로 마친 가등기(假登記) 또는 소유권이전등기(所有權移轉登記)의 효력을 정함을 목적으로 한다.', references=[])

ID
가등기담보등에관한법률_1

LAW
가등기담보등에관한법률

ARTICLE
제1조

TITLE
목적

CLAUSES

REFERENCES
[]


In [ ]:
# ==========================================
# Cell 4
# Search Text + Metadata Builder
# ==========================================

import re
import pickle
from collections import Counter


# --------------------------------------------------
# 법률 키워드 확장 사전
# --------------------------------------------------

LEGAL_EXPANSION = {

    "기망":[
        "속임",
        "허위",
        "기만"
    ],

    "편취":[
        "취득",
        "가로채기"
    ],

    "재산상":[
        "재산",
        "경제적"
    ],

    "위법":[
        "불법",
        "법률위반"
    ],

    "손해":[
        "피해",
        "손실"
    ],

    "계약":[
        "약정",
        "합의"
    ]

}



# --------------------------------------------------
# Kiwi Token
# --------------------------------------------------

def kiwi_extract(text):

    tokens=[]

    result = kiwi.tokenize(text)


    for token in result:

        if token.tag.startswith(
            (
                "N",
                "V"
            )
        ):

            word = token.form.strip()

            if len(word)>=2:

                tokens.append(word)


    return tokens



# --------------------------------------------------
# Keyword Extract
# --------------------------------------------------

def extract_keywords(
    text,
    topk=20
):

    tokens = kiwi_extract(
        text
    )


    counter = Counter(
        tokens
    )


    keywords = [

        x[0]

        for x in counter.most_common(topk)

    ]


    expanded=[]


    for k in keywords:

        expanded.append(k)

        if k in LEGAL_EXPANSION:

            expanded.extend(
                LEGAL_EXPANSION[k]
            )


    return list(
        dict.fromkeys(
            expanded
        )
    )



# --------------------------------------------------
# Display Text
# --------------------------------------------------

def build_display_text(
    article
):

    parts=[]


    parts.append(
        article.article
    )


    if article.title:

        parts.append(
            f"({article.title})"
        )


    parts.append(
        article.body
    )


    return "\n\n".join(
        parts
    )



# --------------------------------------------------
# Search Text
# --------------------------------------------------

def build_search_text(
    article,
    keywords
):

    parts=[]


    parts.append(
        article.law_name
    )


    for x in [

        article.part,

        article.chapter,

        article.section,

        article.subsection

    ]:

        if x:

            parts.append(x)



    parts.append(
        article.article
    )


    if article.title:

        parts.append(
            article.title
        )


    if keywords:

        parts.append(
            "키워드"
        )

        parts.extend(
            keywords
        )


    parts.append(
        "본문"
    )


    parts.append(
        article.body
    )


    return "\n".join(
        parts
    )



# --------------------------------------------------
# Build Documents
# --------------------------------------------------

DOCUMENTS=[]

SEARCH_TEXTS=[]

METADATAS=[]

IDS=[]



for article in tqdm(
    ALL_ARTICLES
):


    keywords = extract_keywords(
        article.body
    )


    display_text = build_display_text(
        article
    )


    search_text = build_search_text(
        article,
        keywords
    )


    metadata={

        "law_name":
            article.law_name,

        "article":
            article.article,

        "title":
            article.title,

        "part":
            article.part,

        "chapter":
            article.chapter,

        "section":
            article.section,

        "subsection":
            article.subsection,

        "references":
            "|".join(
                article.references
            ),

        "keywords":
            "|".join(
                keywords
            ),

        "clause_count":
            len(article.clauses),

        "length":
            len(article.body),

    }



    DOCUMENTS.append(
        display_text
    )

    SEARCH_TEXTS.append(
        search_text
    )

    METADATAS.append(
        metadata
    )

    IDS.append(
        article.id
    )



# --------------------------------------------------
# Save
# --------------------------------------------------

with open(
    CACHE_DIR/"documents.pkl",
    "wb"
) as f:

    pickle.dump(
        {
            "documents":DOCUMENTS,
            "search_texts":SEARCH_TEXTS,
            "metadatas":METADATAS,
            "ids":IDS
        },
        f
    )



print(
    len(DOCUMENTS)
)

print(
    IDS[0]
)

print(
    SEARCH_TEXTS[0][:500]
)

  0%|          | 0/7852 [00:00<?, ?it/s]

7852
가등기담보등에관한법률_1
가등기담보등에관한법률
제1조
목적
키워드
차용
재산
이전
예약
담보
목적
등기
반환
관하
차주
갈음
당시
가액
붙이
이자
합산
액수
초과
경우
따르
본문
이 법은 차용물(借用物)의 반환에 관하여 차주(借主)가 차용물을 갈음하여 다른 재산권을 이전할 것을 예약할 때 그 재산의 예약 당시 가액(價額)이 차용액(借用額)과 이에 붙인 이자를 합산한 액수를 초과하는 경우에 이에 따른 담보계약(擔保契約)과 그 담보의 목적으로 마친 가등기(假登記) 또는 소유권이전등기(所有權移轉登記)의 효력을 정함을 목적으로 한다.


In [ ]:
# ==========================================
# Cell 5
# Reference Graph Builder
# ==========================================

import pickle
from collections import defaultdict, deque


# --------------------------------------------------
# Article map
# --------------------------------------------------

ARTICLE_MAP = {

    article.id: article

    for article in ALL_ARTICLES

}



# --------------------------------------------------
# Reference normalize
# --------------------------------------------------

def normalize_ref(
    law_name,
    ref
):

    number = (
        re.sub(
            r"[^\d의]",
            "",
            ref
        )
    )

    return f"{law_name}_{number}"



# --------------------------------------------------
# Build graph
# --------------------------------------------------

GRAPH = defaultdict(list)

INCOMING_GRAPH = defaultdict(list)



for article in ALL_ARTICLES:


    source = article.id


    for ref in article.references:


        target = normalize_ref(
            article.law_name,
            ref
        )


        if target in ARTICLE_MAP:


            GRAPH[source].append(
                target
            )


            INCOMING_GRAPH[target].append(
                source
            )



# --------------------------------------------------
# remove duplicate
# --------------------------------------------------

for k in GRAPH:

    GRAPH[k] = list(
        set(
            GRAPH[k]
        )
    )


for k in INCOMING_GRAPH:

    INCOMING_GRAPH[k] = list(
        set(
            INCOMING_GRAPH[k]
        )
    )



# --------------------------------------------------
# Graph expansion
# --------------------------------------------------

def expand_graph(
    article_id,
    depth=2
):

    visited=set()

    queue=deque()


    queue.append(
        (
            article_id,
            0
        )
    )


    while queue:


        current, level = queue.popleft()


        if level >= depth:

            continue


        for nxt in GRAPH.get(
            current,
            []
        ):

            if nxt not in visited:

                visited.add(
                    nxt
                )

                queue.append(
                    (
                        nxt,
                        level+1
                    )
                )


        for nxt in INCOMING_GRAPH.get(
            current,
            []
        ):

            if nxt not in visited:

                visited.add(
                    nxt
                )

                queue.append(
                    (
                        nxt,
                        level+1
                    )
                )



    return list(
        visited
    )



# --------------------------------------------------
# Save graph
# --------------------------------------------------

with open(
    CACHE_DIR/"graph.pkl",
    "wb"
) as f:

    pickle.dump(
        {
            "graph":dict(GRAPH),
            "incoming":dict(INCOMING_GRAPH)
        },
        f
    )



print("="*50)

print(
    "Graph Nodes:",
    len(GRAPH)
)

print(
    "Incoming Nodes:",
    len(INCOMING_GRAPH)
)

print("="*50)



# Test

sample_id = ALL_ARTICLES[0].id


print()

print(
    sample_id
)


print(
    expand_graph(
        sample_id
    )
)

Graph Nodes: 3007
Incoming Nodes: 3436

가등기담보등에관한법률_1
[]


In [ ]:
# ==========================================
# Cell 6
# BGE-M3 Embedding
# ==========================================

import pickle
import numpy as np
from tqdm.auto import tqdm


# --------------------------------------------------
# Load Search Data
# --------------------------------------------------

with open(
    CACHE_DIR/"documents.pkl",
    "rb"
) as f:

    data = pickle.load(f)



DOCUMENTS = data["documents"]

SEARCH_TEXTS = data["search_texts"]

METADATAS = data["metadatas"]

IDS = data["ids"]



# --------------------------------------------------
# Embedding
# --------------------------------------------------

print(
    "Embedding documents:",
    len(SEARCH_TEXTS)
)


embeddings = []


BATCH_SIZE = 32



for i in tqdm(
    range(
        0,
        len(SEARCH_TEXTS),
        BATCH_SIZE
    )
):

    batch = SEARCH_TEXTS[
        i:i+BATCH_SIZE
    ]


    emb = embedding_model.encode(

        batch,

        batch_size=BATCH_SIZE,

        normalize_embeddings=True,

        show_progress_bar=False

    )


    embeddings.extend(
        emb
    )



embeddings = np.array(
    embeddings
)



print(
    embeddings.shape
)



# --------------------------------------------------
# Save
# --------------------------------------------------

with open(
    CACHE_DIR/"embeddings.pkl",
    "wb"
) as f:

    pickle.dump(
        embeddings,
        f
    )


print(
    "Embedding Saved"
)

Embedding documents: 7852


  0%|          | 0/246 [00:00<?, ?it/s]

(7852, 1024)
Embedding Saved


In [ ]:
# ==========================================
# Cell 7
# ChromaDB Storage
# ==========================================

import pickle
from tqdm.auto import tqdm


# --------------------------------------------------
# Load
# --------------------------------------------------

with open(
    CACHE_DIR/"embeddings.pkl",
    "rb"
) as f:

    embeddings = pickle.load(f)



with open(
    CACHE_DIR/"documents.pkl",
    "rb"
) as f:

    data = pickle.load(f)



DOCUMENTS = data["documents"]

METADATAS = data["metadatas"]

IDS = data["ids"]



# --------------------------------------------------
# Chroma Insert
# --------------------------------------------------

print(
    "Adding to Chroma..."
)


BATCH_SIZE = 500



for i in tqdm(
    range(
        0,
        len(IDS),
        BATCH_SIZE
    )
):

    end = min(
        i+BATCH_SIZE,
        len(IDS)
    )


    collection.add(

        ids=IDS[i:end],

        documents=DOCUMENTS[i:end],

        embeddings=
            embeddings[i:end].tolist(),

        metadatas=
            METADATAS[i:end]

    )



print()

print("="*50)

print(
    "Chroma Count:",
    collection.count()
)

print("="*50)



# --------------------------------------------------
# Persist info
# --------------------------------------------------

with open(
    CACHE_DIR/"chroma_ready.pkl",
    "wb"
) as f:

    pickle.dump(
        {
            "count":collection.count()
        },
        f
    )


print(
    "ChromaDB Ready"
)

Adding to Chroma...


  0%|          | 0/16 [00:00<?, ?it/s]


Chroma Count: 7852
ChromaDB Ready


In [ ]:
# ==========================================
# Cell 8
# BM25 Index Build
# ==========================================

import pickle
from rank_bm25 import BM25Okapi
from tqdm.auto import tqdm


# --------------------------------------------------
# Load
# --------------------------------------------------

with open(
    CACHE_DIR/"documents.pkl",
    "rb"
) as f:

    data = pickle.load(f)



SEARCH_TEXTS = data["search_texts"]

IDS = data["ids"]



# --------------------------------------------------
# Tokenizer
# --------------------------------------------------

def bm25_tokenize(text):

    tokens=[]


    for token in kiwi.tokenize(text):

        if token.tag.startswith(
            (
                "N",
                "V"
            )
        ):

            word = token.form.strip()

            if len(word)>=2:

                tokens.append(word)


    return tokens



# --------------------------------------------------
# Build Tokens
# --------------------------------------------------

print(
    "Building BM25 Tokens..."
)


BM25_TOKENS=[]


for text in tqdm(
    SEARCH_TEXTS
):

    BM25_TOKENS.append(
        bm25_tokenize(text)
    )



# --------------------------------------------------
# BM25
# --------------------------------------------------

bm25 = BM25Okapi(
    BM25_TOKENS
)



# --------------------------------------------------
# Save
# --------------------------------------------------

with open(
    INDEX_DIR/"bm25.pkl",
    "wb"
) as f:

    pickle.dump(
        {
            "bm25":bm25,
            "tokens":BM25_TOKENS,
            "ids":IDS
        },
        f
    )



print()

print(
    "BM25 Ready"
)

print(
    "Documents:",
    len(BM25_TOKENS)
)

Building BM25 Tokens...


  0%|          | 0/7852 [00:00<?, ?it/s]


BM25 Ready
Documents: 7852


In [ ]:
# ==========================================
# Cell 9
# Hybrid Retriever
# Dense + BM25 + Graph + RRF
# ==========================================

import pickle
import numpy as np
from collections import defaultdict


# --------------------------------------------------
# Load BM25
# --------------------------------------------------

with open(
    INDEX_DIR/"bm25.pkl",
    "rb"
) as f:

    bm25_data = pickle.load(f)


bm25 = bm25_data["bm25"]

BM25_IDS = bm25_data["ids"]



# --------------------------------------------------
# Load Graph
# --------------------------------------------------

with open(
    CACHE_DIR/"graph.pkl",
    "rb"
) as f:

    graph_data = pickle.load(f)



GRAPH = graph_data["graph"]

INCOMING_GRAPH = graph_data["incoming"]



# --------------------------------------------------
# ID map
# --------------------------------------------------

ID_TO_INDEX = {

    x:i

    for i,x in enumerate(
        IDS
    )

}



# --------------------------------------------------
# BM25 Search
# --------------------------------------------------

def bm25_search(
    query,
    top_k=40
):

    tokens = bm25_tokenize(
        query
    )


    scores = bm25.get_scores(
        tokens
    )


    idx = np.argsort(
        scores
    )[::-1][:top_k]


    return [

        (
            IDS[i],
            float(scores[i])
        )

        for i in idx

    ]



# --------------------------------------------------
# Dense Search
# --------------------------------------------------

def dense_search(
    query,
    top_k=40
):

    q_emb = embedding_model.encode(

        query,

        normalize_embeddings=True

    )


    result = collection.query(

        query_embeddings=[
            q_emb.tolist()
        ],

        n_results=top_k

    )


    output=[]


    for i,doc_id in enumerate(
        result["ids"][0]
    ):

        distance = result["distances"][0][i]

        score = 1-distance


        output.append(
            (
                doc_id,
                float(score)
            )
        )


    return output



# --------------------------------------------------
# RRF
# --------------------------------------------------

def reciprocal_rank_fusion(
    results,
    k=60
):

    scores=defaultdict(float)


    for result in results:

        for rank,(doc_id,score) in enumerate(result):

            scores[doc_id]+=(
                1/(k+rank+1)
            )


    return sorted(

        scores.items(),

        key=lambda x:x[1],

        reverse=True

    )



# --------------------------------------------------
# Graph Expand
# --------------------------------------------------

def graph_expand(
    doc_ids,
    depth=1
):

    expanded=set()


    for doc_id in doc_ids:

        expanded.add(
            doc_id
        )


        for x in GRAPH.get(
            doc_id,
            []
        ):

            expanded.add(x)


        for x in INCOMING_GRAPH.get(
            doc_id,
            []
        ):

            expanded.add(x)



    return list(expanded)



# --------------------------------------------------
# Hybrid Search
# --------------------------------------------------

def hybrid_search(
    query,
    top_k=10
):


    dense = dense_search(
        query,
        40
    )


    sparse = bm25_search(
        query,
        40
    )



    fused = reciprocal_rank_fusion(
        [
            dense,
            sparse
        ]
    )



    candidates = [

        x[0]

        for x in fused[:20]

    ]



    expanded = graph_expand(
        candidates
    )



    final=[]


    for doc_id in expanded:


        if doc_id in ID_TO_INDEX:

            idx = ID_TO_INDEX[doc_id]

            final.append(

                (
                    doc_id,

                    fused[0][1]
                    if doc_id in dict(fused)
                    else 0

                )

            )



    final = sorted(

        final,

        key=lambda x:x[1],

        reverse=True

    )


    return final[:top_k]



print(
    "Retriever Ready"
)

Retriever Ready


In [ ]:
# ==========================================
# Cell 10
# Retriever Test
# ==========================================

import pickle


# --------------------------------------------------
# Result Formatter
# --------------------------------------------------

with open(
    CACHE_DIR/"documents.pkl",
    "rb"
) as f:

    data = pickle.load(f)


DOCUMENTS = data["documents"]


ID_TO_DOC = {

    id_:doc

    for id_,doc in zip(
        IDS,
        DOCUMENTS
    )

}


ID_TO_META = {

    id_:meta

    for id_,meta in zip(
        IDS,
        METADATAS
    )

}



def search_test(
    query,
    top_k=10
):


    results = hybrid_search(
        query,
        top_k
    )


    print()
    print("="*70)
    print("QUERY")
    print(query)
    print("="*70)



    for rank,(doc_id,score) in enumerate(
        results,
        1
    ):


        meta = ID_TO_META.get(
            doc_id,
            {}
        )


        print()
        print(
            f"[{rank}]",
            doc_id,
            "score=",
            round(score,4)
        )


        print(
            meta.get(
                "law_name",
                ""
            ),
            meta.get(
                "article",
                ""
            ),
            meta.get(
                "title",
                ""
            )
        )


        print(
            ID_TO_DOC[doc_id][:300]
        )


        print("-"*70)



# --------------------------------------------------
# Test Queries
# --------------------------------------------------

queries = [

    "사람을 속여 돈을 받은 경우 사기죄",

    "임차인의 계약갱신 요구",

    "공무원의 직무상 불법행위 국가배상",

    "타인의 재물을 훔친 경우",

]


for q in queries:

    search_test(
        q,
        top_k=5
    )


QUERY
사람을 속여 돈을 받은 경우 사기죄

[1] 도로교통법_73 score= 0.0328
도로교통법 제73조 교통안전교육
제73조

(교통안전교육)

**①** 운전면허를 받으려는 사람은 대통령령으로 정하는 바에 따라 제83조제1항제2호와 제3호에 따른 시험에 응시하기 전에 다음 각 호의 사항에 관한 교통안전교육을 받아야 한다. 다만, 제2항제1호에 따라 특별교통안전 의무교육을 받은 사람 또는 제104조제1항에 따른 자동차운전 전문학원에서 학과교육을 수료한 사람은 그러하지 아니하다. <개정 2014.12.30, 2017.10.24, 2018.3.27>
1\. 운전자가 갖추어야 하는 기본예절
2\. 도로교통에 관한 법령과 지식
3\. 안전운전 능력
3의
----------------------------------------------------------------------

[2] 형법_357 score= 0.0328
형법 제357조 배임수증재
제357조

(배임수증재)

**①** 타인의 사무를 처리하는 자가 그 임무에 관하여 부정한 청탁을 받고 재물 또는 재산상의 이익을 취득하거나 제3자로 하여금 이를 취득하게 한 때에는 5년 이하의 징역 또는 1천만원 이하의 벌금에 처한다. <개정 2016.5.29>
**②** 제1항의 재물 또는 재산상 이익을 공여한 자는 2년 이하의 징역 또는 500만원 이하의 벌금에 처한다. <개정 2020.12.8>
**③** 범인 또는 그 사정을 아는 제3자가 취득한 제1항의 재물은 몰수한다. 그 재물을 몰수하기 불가능하거나 재산상의 이익을 
----------------------------------------------------------------------

[3] 도로교통법_156 score= 0.0328
도로교통법 제156조 벌칙
제156조

(벌칙)

다음 각 호의 어느 하나에 해당하는 사람은 20만원 이하의 벌금이나 구류 또는 과료(科料)에 처한다. <개정 2013.8.13, 2014.1.28, 20

In [ ]:
# ==========================================
# Cell 11
# Advanced Retriever Tuning
# Query Expansion + Graph Boost + Metadata Filter
# ==========================================

from collections import defaultdict
import numpy as np



# --------------------------------------------------
# Query Expansion
# --------------------------------------------------

def expand_query(
    query
):

    keywords = extract_keywords(
        query,
        topk=10
    )


    expanded=[]


    for k in keywords:

        expanded.append(k)


        if k in LEGAL_EXPANSION:

            expanded.extend(
                LEGAL_EXPANSION[k]
            )


    return list(
        dict.fromkeys(
            expanded
        )
    )



# --------------------------------------------------
# Query Builder
# --------------------------------------------------

def build_query_text(
    query
):

    expanded = expand_query(
        query
    )


    return "\n".join(

        [
            query,
            " ".join(expanded)
        ]

    )



# --------------------------------------------------
# Improved Dense Search
# --------------------------------------------------

def dense_search_v2(
    query,
    top_k=50
):

    query_text = build_query_text(
        query
    )


    q_emb = embedding_model.encode(

        query_text,

        normalize_embeddings=True

    )



    result = collection.query(

        query_embeddings=[
            q_emb.tolist()
        ],

        n_results=top_k

    )


    output=[]


    for i,doc_id in enumerate(
        result["ids"][0]
    ):

        score = (
            1 -
            result["distances"][0][i]
        )


        output.append(
            (
                doc_id,
                float(score)
            )
        )


    return output



# --------------------------------------------------
# Graph Boost
# --------------------------------------------------

def apply_graph_boost(
    results,
    boost=0.05
):

    scores = defaultdict(float)


    for doc_id,score in results:

        scores[doc_id]+=score



    base_ids=list(
        scores.keys()
    )


    related = graph_expand(
        base_ids,
        depth=1
    )


    for doc_id in related:

        if doc_id not in scores:

            scores[doc_id]=boost


        else:

            scores[doc_id]+=boost



    return sorted(

        scores.items(),

        key=lambda x:x[1],

        reverse=True

    )



# --------------------------------------------------
# Final Retriever
# --------------------------------------------------

def legal_search(
    query,
    top_k=10
):


    dense = dense_search_v2(
        query,
        50
    )


    sparse = bm25_search(
        query,
        50
    )



    fused = reciprocal_rank_fusion(

        [
            dense,
            sparse
        ]

    )



    boosted = apply_graph_boost(
        fused
    )



    return boosted[:top_k]



print(
    "Advanced Retriever Ready"
)

Advanced Retriever Ready


In [ ]:
# ==========================================
# Cell 12
# Final Search Interface
# ==========================================

import pickle


# --------------------------------------------------
# Load display data
# --------------------------------------------------

with open(
    CACHE_DIR/"documents.pkl",
    "rb"
) as f:

    data = pickle.load(f)


DISPLAY_DOCS = data["documents"]

IDS = data["ids"]

METAS = data["metadatas"]



DOC_MAP = {

    i:d

    for i,d in zip(
        IDS,
        DISPLAY_DOCS
    )

}


META_MAP = {

    i:m

    for i,m in zip(
        IDS,
        METAS
    )

}



# --------------------------------------------------
# Pretty Search
# --------------------------------------------------

def law_search(
    query,
    top_k=10
):


    results = legal_search(
        query,
        top_k
    )


    print()
    print("="*80)

    print(
        "QUERY:",
        query
    )

    print("="*80)



    for rank,(doc_id,score) in enumerate(
        results,
        1
    ):


        meta = META_MAP.get(
            doc_id,
            {}
        )


        print()

        print(
            f"[{rank}]",
            doc_id,
            "score:",
            round(score,4)
        )


        print(
            "법령:",
            meta.get(
                "law_name",
                ""
            )
        )


        print(
            "조문:",
            meta.get(
                "article",
                ""
            ),
            meta.get(
                "title",
                ""
            )
        )


        print()

        print(
            DOC_MAP[doc_id][:500]
        )


        print("-"*80)



# --------------------------------------------------
# Example
# --------------------------------------------------

law_search(
    "사람을 속여 재물을 받은 경우 처벌"
)



law_search(
    "임대인이 계약 갱신을 거절할 수 있는 경우"
)



law_search(
    "공무원의 불법행위로 손해가 발생한 경우"
)


QUERY: 사람을 속여 재물을 받은 경우 처벌

[1] 형법_347 score: 0.0828
법령: 형법
조문: 제347조 사기

제347조

(사기)

**①** 사람을 기망하여 재물의 교부를 받거나 재산상의 이익을 취득한 자는 20년 이하의 징역 또는 5천만원 이하의 벌금에 처한다. <개정 2025.12.23>
**②** 전항의 방법으로 제삼자로 하여금 재물의 교부를 받게 하거나 재산상의 이익을 취득하게 한 때에도 전항의 형과 같다.
--------------------------------------------------------------------------------

[2] 형법_350 score: 0.082
법령: 형법
조문: 제350조 공갈

제350조

(공갈)

**①** 사람을 공갈하여 재물의 교부를 받거나 재산상의 이익을 취득한 자는 10년 이하의 징역 또는 2천만원 이하의 벌금에 처한다. <개정 1995.12.29>
**②** 전항의 방법으로 제삼자로 하여금 재물의 교부를 받게 하거나 재산상의 이익을 취득하게 한 때에도 전항의 형과 같다.
--------------------------------------------------------------------------------

[3] 형법_348 score: 0.0803
법령: 형법
조문: 제348조 준사기

제348조

(준사기)

**①** 미성년자의 사리분별력 부족 또는 사람의 심신장애를 이용하여 재물을 교부받거나 재산상 이익을 취득한 자는 20년 이하의 징역 또는 5천만원 이하의 벌금에 처한다. <개정 2025.12.23>
**②** 제1항의 방법으로 제3자로 하여금 재물을 교부받게 하거나 재산상 이익을 취득하게 한 경우에도 제1항의 형에 처한다.
--------------------------------------------------------------------------------

[4] 형법_357 score: 0.0795
법령: 형법
조문: 제357조 

In [ ]:
# ==========================================
# Cell 13
# Retrieval Debug & Score Analysis
# ==========================================

from collections import defaultdict


def debug_search(
    query,
    top_k=20
):

    dense = dense_search_v2(
        query,
        top_k
    )

    sparse = bm25_search(
        query,
        top_k
    )


    fused = reciprocal_rank_fusion(
        [
            dense,
            sparse
        ]
    )


    boosted = apply_graph_boost(
        fused
    )


    print("="*80)
    print("QUERY")
    print(query)
    print("="*80)


    print("\n[DENSE]")
    for rank,(doc,score) in enumerate(dense[:10],1):
        print(
            rank,
            doc,
            round(score,4)
        )


    print("\n[BM25]")
    for rank,(doc,score) in enumerate(sparse[:10],1):
        print(
            rank,
            doc,
            round(score,4)
        )


    print("\n[RRF]")
    for rank,(doc,score) in enumerate(fused[:10],1):
        print(
            rank,
            doc,
            round(score,4)
        )


    print("\n[GRAPH BOOST]")
    for rank,(doc,score) in enumerate(boosted[:10],1):
        print(
            rank,
            doc,
            round(score,4)
        )



debug_search(
    "사람을 속여 재산을 취득한 경우"
)

QUERY
사람을 속여 재산을 취득한 경우

[DENSE]
1 형법_347 0.6396
2 형법_350 0.6048
3 특정범죄가중처벌등에관한법률_13 0.5999
4 형법_357 0.5896
5 형법_347의2 0.5845
6 형법_349 0.5844
7 형법_336 0.5836
8 민법_249 0.5664
9 형법_48 0.5642
10 형법_348 0.5634

[BM25]
1 형법_347 23.4534
2 형법_350 13.8668
3 형법_336 13.6475
4 형법_348 13.1211
5 특정범죄가중처벌등에관한법률_12 13.055
6 특정경제범죄가중처벌등에관한법률_3 11.1912
7 민법_248 10.7899
8 부동산등기법_111 10.1961
9 민법_944 10.0565
10 형법_357 9.9081

[RRF]
1 형법_347 0.0328
2 형법_350 0.0323
3 형법_336 0.0308
4 형법_357 0.0299
5 형법_348 0.0299
6 특정범죄가중처벌등에관한법률_13 0.0289
7 형법_349 0.0289
8 특정경제범죄가중처벌등에관한법률_3 0.0283
9 민법_248 0.0274
10 민법_944 0.0273

[GRAPH BOOST]
1 형법_347 0.0828
2 형법_350 0.0823
3 형법_336 0.0808
4 형법_357 0.0799
5 형법_348 0.0799
6 특정범죄가중처벌등에관한법률_13 0.0789
7 형법_349 0.0789
8 특정경제범죄가중처벌등에관한법률_3 0.0783
9 민법_248 0.0774
10 민법_944 0.0773


In [ ]:
# ==========================================
# Cell 14
# Persistent Reload System
# ==========================================

import pickle
import os
from chromadb import PersistentClient



# --------------------------------------------------
# Reload Chroma
# --------------------------------------------------

client_reload = PersistentClient(
    path=DB_PATH
)


collection_reload = client_reload.get_collection(
    name=COLLECTION_NAME
)



# --------------------------------------------------
# Reload BM25
# --------------------------------------------------

with open(
    INDEX_DIR/"bm25.pkl",
    "rb"
) as f:

    bm25_data = pickle.load(f)



bm25 = bm25_data["bm25"]

BM25_TOKENS = bm25_data["tokens"]

BM25_IDS = bm25_data["ids"]



# --------------------------------------------------
# Reload Graph
# --------------------------------------------------

with open(
    CACHE_DIR/"graph.pkl",
    "rb"
) as f:

    graph_data = pickle.load(f)



GRAPH = graph_data["graph"]

INCOMING_GRAPH = graph_data["incoming"]



# --------------------------------------------------
# Reload Documents
# --------------------------------------------------

with open(
    CACHE_DIR/"documents.pkl",
    "rb"
) as f:

    doc_data = pickle.load(f)



DOCUMENTS = doc_data["documents"]

SEARCH_TEXTS = doc_data["search_texts"]

METADATAS = doc_data["metadatas"]

IDS = doc_data["ids"]



ID_TO_INDEX = {

    x:i

    for i,x in enumerate(IDS)

}


ID_TO_META = {

    x:m

    for x,m in zip(
        IDS,
        METADATAS
    )

}


ID_TO_DOC = {

    x:d

    for x,d in zip(
        IDS,
        DOCUMENTS
    )

}



print("="*60)

print(
    "Reload Complete"
)

print(
    "Chroma:",
    collection_reload.count()
)

print(
    "Documents:",
    len(DOCUMENTS)
)

print(
    "Graph:",
    len(GRAPH)
)

print("="*60)

NameError: name 'DB_PATH' is not defined

In [ ]:
# ==========================================
# Cell 15
# Legal Issue Extractor
# ==========================================

import re
from collections import Counter



# --------------------------------------------------
# Issue Dictionary
# --------------------------------------------------

ISSUE_DICTIONARY = {

    "사기":[
        "기망",
        "처분행위",
        "재산상 이익",
        "편취",
        "재물"
    ],

    "횡령":[
        "보관",
        "위탁",
        "불법영득의사",
        "반환거부"
    ],

    "배임":[
        "임무위배",
        "재산상 손해",
        "타인의 사무"
    ],

    "절도":[
        "절취",
        "타인의 재물",
        "영득"
    ],

    "손해배상":[
        "불법행위",
        "고의",
        "과실",
        "손해",
        "인과관계"
    ],

    "계약":[
        "청약",
        "승낙",
        "의사표시",
        "해제",
        "해지"
    ],

    "임대차":[
        "임차인",
        "임대인",
        "갱신",
        "보증금"
    ],

    "헌법":[
        "기본권",
        "평등권",
        "자유권",
        "비례원칙"
    ]

}



# --------------------------------------------------
# Extract Issues
# --------------------------------------------------

def extract_issues(
    query
):

    found=[]


    query_lower=query


    for issue,terms in ISSUE_DICTIONARY.items():


        score=0


        if issue in query_lower:

            score+=3


        for t in terms:

            if t in query_lower:

                score+=1



        if score>0:

            found.append(
                (
                    issue,
                    score
                )
            )


    found.sort(
        key=lambda x:x[1],
        reverse=True
    )


    return [
        x[0]
        for x in found
    ]



# --------------------------------------------------
# Query Builder
# --------------------------------------------------

def legal_query_builder(
    query
):


    issues = extract_issues(
        query
    )


    expanded=[]


    for issue in issues:

        expanded.append(
            issue
        )

        expanded.extend(
            ISSUE_DICTIONARY[issue]
        )


    return {

        "original":
            query,

        "issues":
            issues,

        "expanded":
            list(
                dict.fromkeys(
                    expanded
                )
            ),

        "search_text":
            query
            +
            "\n"
            +
            " ".join(expanded)

    }



# --------------------------------------------------
# Test
# --------------------------------------------------

result = legal_query_builder(
    "A가 거짓말하여 돈을 받은 경우 사기죄가 되는가"
)


print(result)

In [ ]:
# ==========================================
# Cell 16
# Reranker (Cross Encoder)
# ==========================================

!pip -q install sentence-transformers


from sentence_transformers import CrossEncoder
import numpy as np



# --------------------------------------------------
# Load Reranker
# --------------------------------------------------

print(
    "Loading Reranker..."
)


reranker = CrossEncoder(
    "BAAI/bge-reranker-v2-m3",
    device=DEVICE
)


print(
    "Reranker Ready"
)



# --------------------------------------------------
# Candidate Retriever
# --------------------------------------------------

def retrieve_candidates(
    query,
    dense_k=50,
    bm25_k=50
):


    dense = dense_search_v2(
        query,
        dense_k
    )


    sparse = bm25_search(
        query,
        bm25_k
    )


    fused = reciprocal_rank_fusion(
        [
            dense,
            sparse
        ]
    )


    return [
        x[0]
        for x in fused[:30]
    ]



# --------------------------------------------------
# Rerank
# --------------------------------------------------

def rerank_results(
    query,
    candidates,
    top_k=10
):


    pairs=[]


    for doc_id in candidates:


        text = ID_TO_DOC.get(
            doc_id,
            ""
        )


        pairs.append(
            [
                query,
                text
            ]
        )



    scores = reranker.predict(
        pairs
    )



    ranked = sorted(

        zip(
            candidates,
            scores
        ),

        key=lambda x:x[1],

        reverse=True

    )



    return ranked[:top_k]



# --------------------------------------------------
# Full Pipeline
# --------------------------------------------------

def advanced_legal_search(
    query,
    top_k=10
):


    query_info = legal_query_builder(
        query
    )


    candidates = retrieve_candidates(
        query_info["search_text"]
    )


    reranked = rerank_results(
        query,
        candidates,
        top_k
    )


    return reranked



# --------------------------------------------------
# Test
# --------------------------------------------------

results = advanced_legal_search(
    "사람을 속여 돈을 받은 경우 사기죄가 성립하는가",
    5
)



for rank,(doc,score) in enumerate(
    results,
    1
):

    print(
        rank,
        doc,
        score
    )

    print(
        ID_TO_DOC[doc][:300]
    )

    print("-"*50)

In [ ]:
# ==========================================
# Cell 17
# Advanced Graph Expansion
# Multi Relation Graph
# ==========================================

from collections import defaultdict, deque
import pickle



# --------------------------------------------------
# Build Extended Graph
# --------------------------------------------------

EXT_GRAPH = defaultdict(set)



for article in ALL_ARTICLES:


    source = article.id


    # 1. Reference Edge

    for ref in article.references:

        target = normalize_ref(
            article.law_name,
            ref
        )


        if target in ARTICLE_MAP:

            EXT_GRAPH[source].add(
                target
            )


            EXT_GRAPH[target].add(
                source
            )



    # 2. Same chapter relation

    for other in ALL_ARTICLES:


        if (
            article.id != other.id
            and
            article.law_name == other.law_name
            and
            article.chapter
            and
            article.chapter == other.chapter
        ):

            EXT_GRAPH[source].add(
                other.id
            )



# --------------------------------------------------
# Keyword Relation
# --------------------------------------------------

KEYWORD_MAP = defaultdict(list)



for article,meta in zip(
    ALL_ARTICLES,
    METADATAS
):

    keywords = meta.get(
        "keywords",
        ""
    ).split("|")


    for k in keywords:

        if k:

            KEYWORD_MAP[k].append(
                article.id
            )



for keyword,docs in KEYWORD_MAP.items():

    if len(docs)<30:

        for a in docs:

            for b in docs:

                if a!=b:

                    EXT_GRAPH[a].add(
                        b
                    )



# --------------------------------------------------
# Convert list
# --------------------------------------------------

EXT_GRAPH = {

    k:list(v)

    for k,v in EXT_GRAPH.items()

}



# --------------------------------------------------
# Expansion
# --------------------------------------------------

def advanced_graph_expand(
    ids,
    depth=2,
    max_nodes=100
):


    visited=set(ids)

    queue=deque()



    for x in ids:

        queue.append(
            (
                x,
                0
            )
        )



    while queue:


        current,level = queue.popleft()


        if level>=depth:

            continue



        for nxt in EXT_GRAPH.get(
            current,
            []
        ):


            if nxt not in visited:


                visited.add(
                    nxt
                )


                queue.append(
                    (
                        nxt,
                        level+1
                    )
                )



        if len(visited)>=max_nodes:

            break



    return list(
        visited
    )



# --------------------------------------------------
# Save
# --------------------------------------------------

with open(
    CACHE_DIR/"extended_graph.pkl",
    "wb"
) as f:

    pickle.dump(
        EXT_GRAPH,
        f
    )



print(
    "Extended Graph Nodes:",
    len(EXT_GRAPH)
)

In [ ]:
# ==========================================
# Cell 18
# RAG Context Builder
# ==========================================

import textwrap



# --------------------------------------------------
# Build Legal Context
# --------------------------------------------------

def build_rag_context(
    query,
    top_k=8
):


    results = advanced_legal_search(
        query,
        top_k
    )


    context=[]


    for rank,(doc_id,score) in enumerate(
        results,
        1
    ):


        meta = ID_TO_META.get(
            doc_id,
            {}
        )


        block = f"""
[관련 법령 {rank}]

법령명:
{meta.get('law_name','')}

조문:
{meta.get('article','')} {meta.get('title','')}

본문:
{ID_TO_DOC.get(doc_id,'')}

검색점수:
{score}
"""


        context.append(
            block.strip()
        )



    return "\n\n".join(
        context
    )



# --------------------------------------------------
# Test
# --------------------------------------------------

query = """
A가 B에게 거짓말하여 돈을 받았다.
사기죄가 성립하는가?
"""


context = build_rag_context(
    query
)


print(
    context[:3000]
)

In [ ]:
# ==========================================
# Cell 19
# Legal Answer Prompt Builder
# ==========================================

import datetime



# --------------------------------------------------
# Prompt Template
# --------------------------------------------------

LEGAL_PROMPT_TEMPLATE = """

당신은 대한민국 변호사시험 답안 작성을 보조하는 법률 AI이다.

반드시 아래 법령 자료만 근거로 답변한다.
법령 자료에 없는 내용은 추측하지 않는다.


========================
문제
========================

{question}


========================
관련 법령
========================

{context}


========================
답안 작성 방식
========================

다음 형식으로 작성한다.


1. 쟁점의 정리

- 문제되는 법적 쟁점을 명확히 제시


2. 관련 법령

- 적용 조문 제시
- 조문의 요건 정리


3. 법리 검토

- 구성요건
- 요건별 판단
- 사안 적용


4. 결론

- 최종 판단


답변은 대한민국 법학전문대학원 시험 답안 스타일로 작성한다.

"""



# --------------------------------------------------
# Build Prompt
# --------------------------------------------------

def build_answer_prompt(
    question,
    top_k=8
):


    context = build_rag_context(
        question,
        top_k
    )


    prompt = LEGAL_PROMPT_TEMPLATE.format(

        question=question,

        context=context

    )


    return prompt



# --------------------------------------------------
# Test
# --------------------------------------------------

question = """

A는 B에게 회사 투자금이라고 속이고
돈을 받아 개인적으로 사용하였다.
A의 형사책임은?

"""


prompt = build_answer_prompt(
    question
)


print(
    prompt[:5000]
)

In [ ]:
# ==========================================
# Cell 20
# Evaluation Dataset Framework
# Retrieval Accuracy Test
# ==========================================

import json
import pickle
from pathlib import Path



# --------------------------------------------------
# Evaluation Data Format
# --------------------------------------------------

EVAL_FILE = CACHE_DIR / "eval_dataset.json"



# 예시 형식
#
# [
#   {
#     "question":
#       "사람을 속여 돈을 받은 경우 사기죄",
#
#     "expected":
#       [
#          "형법_347"
#       ]
#   }
# ]



if not EVAL_FILE.exists():

    sample_eval = [

        {

            "question":
            "사람을 기망하여 재물을 받은 경우 사기죄",

            "expected":
            [
                "형법_347"
            ]

        },

        {

            "question":
            "타인의 재물을 훔친 경우",

            "expected":
            [
                "형법_329"
            ]

        },

        {

            "question":
            "임대인이 임차인의 계약 갱신 요구를 거절할 수 있는 경우",

            "expected":
            [
                "주택임대차보호법"
            ]

        }

    ]


    with open(
        EVAL_FILE,
        "w",
        encoding="utf8"
    ) as f:

        json.dump(
            sample_eval,
            f,
            ensure_ascii=False,
            indent=2
        )



# --------------------------------------------------
# Load Evaluation
# --------------------------------------------------

with open(
    EVAL_FILE,
    encoding="utf8"
) as f:

    EVAL_DATA = json.load(f)



# --------------------------------------------------
# Metrics
# --------------------------------------------------

def evaluate_retrieval(
    dataset,
    top_k=10
):


    results=[]


    hit=0


    for item in dataset:


        query = item["question"]

        expected = item["expected"]



        retrieved = advanced_legal_search(
            query,
            top_k
        )


        retrieved_ids = [

            x[0]

            for x in retrieved

        ]



        success=False


        for target in expected:


            for rid in retrieved_ids:

                if target in rid:

                    success=True



        if success:

            hit += 1



        results.append(

            {

                "question":
                    query,

                "success":
                    success,

                "retrieved":
                    retrieved_ids

            }

        )



    score = hit / len(dataset)



    return {

        "recall@k":
            score,

        "total":
            len(dataset),

        "results":
            results

    }



# --------------------------------------------------
# Run
# --------------------------------------------------

evaluation_result = evaluate_retrieval(
    EVAL_DATA,
    top_k=10
)


print(
    "Recall@10:",
    evaluation_result["recall@k"]
)


for r in evaluation_result["results"]:

    print()

    print(
        r["question"]
    )

    print(
        r["success"]
    )

In [ ]:
# ==========================================
# Cell 21
# Final Legal RAG Pipeline
# solve_law_problem()
# ==========================================

import json
import time



# --------------------------------------------------
# Final Pipeline
# --------------------------------------------------

def solve_law_problem(
    question,
    top_k=8,
    return_prompt=False
):


    start = time.time()



    # 1. Query Analysis

    query_info = legal_query_builder(
        question
    )



    # 2. Retrieval

    results = advanced_legal_search(
        query_info["search_text"],
        top_k
    )



    # 3. Context

    context=[]


    for rank,(doc_id,score) in enumerate(
        results,
        1
    ):


        meta = ID_TO_META.get(
            doc_id,
            {}
        )


        context.append(

            {

                "rank":
                    rank,

                "id":
                    doc_id,

                "score":
                    float(score),

                "law":
                    meta.get(
                        "law_name",
                        ""
                    ),

                "article":
                    meta.get(
                        "article",
                        ""
                    ),

                "title":
                    meta.get(
                        "title",
                        ""
                    ),

                "text":
                    ID_TO_DOC.get(
                        doc_id,
                        ""
                    )

            }

        )



    # 4. Prompt

    prompt = build_answer_prompt(
        question,
        top_k
    )



    elapsed = time.time()-start



    result = {


        "question":
            question,


        "issues":
            query_info["issues"],


        "retrieved_documents":
            context,


        "prompt":
            prompt,


        "elapsed":
            elapsed

    }



    if return_prompt:

        return result



    return result



# --------------------------------------------------
# Display
# --------------------------------------------------

def print_solution_result(
    result
):


    print("="*80)

    print("QUESTION")

    print(result["question"])

    print("="*80)



    print()

    print("ISSUES")

    for x in result["issues"]:

        print(
            "-",
            x
        )



    print()

    print("RETRIEVED LAW")


    for doc in result["retrieved_documents"]:


        print()

        print(
            "["+str(doc["rank"])+"]",
            doc["law"],
            doc["article"],
            doc["title"]
        )


        print(
            doc["text"][:400]
        )

        print("-"*50)



    print()

    print(
        "TIME:",
        round(
            result["elapsed"],
            3
        ),
        "sec"
    )



# --------------------------------------------------
# Test
# --------------------------------------------------

answer = solve_law_problem(

    """
    A는 B에게 거짓말하여 돈을 받았다.
    A에게 사기죄가 성립하는가?
    """,

    top_k=5

)


print_solution_result(
    answer
)

In [ ]:
import shutil

shutil.make_archive(
    "/content/legal_rag_full_backup",
    "zip",
    "/working"
)

'/content/legal_rag_full_backup.zip'

In [ ]:
from google.colab import files

files.download(
    "/content/legal_rag_full_backup.zip"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>